# Padding Mask

배치 안에서 문장 길이를 맞추려고 짧은 문장 뒤에 채워 넣은 `<pad>` 토큰은 실제 의미가 없다. 그래서 Attention을 계산할 때 `<pad>` 위치는 완전히 무시하도록 가려야 한다.

가리는 방법은 3주차 Self-Attention 때부터 `scaled_dot_product_attention`에 이미 있던 `mask` 인자를 실제로 채워서 넘기는 것이다.

**scores = scores.masked_fill(mask == 0, -inf) → softmax(scores)**

* `mask`는 실제 토큰이면 1(True), `<pad>`면 0(False)
* 값을 아예 지우지 않고 `-inf`로 바꾸는 이유: softmax(-inf) = 0이 되면서도, 행렬 shape은 그대로 유지되어 배치 연산이 깨지지 않기 때문

In [ ]:
import torch
import torch.nn as nn

# seq: [batch, seq_len] 토큰 id, pad_idx는 <pad> 토큰의 id (보통 0)
def make_padding_mask(seq, pad_idx=0):
    mask = (seq != pad_idx).unsqueeze(1).unsqueeze(2)
    return mask   # [batch, 1, 1, seq_len] — attention score에 브로드캐스팅됨

seq = torch.tensor([[5, 8, 3, 0, 0]])
print(make_padding_mask(seq))
# tensor([[[[ True,  True,  True, False, False]]]])

# Look-ahead Mask (Causal Mask)

지금까지(4~5주차) 만든 건 Encoder뿐이라 문장 전체를 한꺼번에 봐도 문제가 없었다. 하지만 06번에서 만들 **Decoder**는 "이전 단어들로 다음 단어를 하나씩 예측"하는 역할이라, 자기보다 뒤에 있는(미래) 정답 단어를 미리 보면 반칙이다.

그래서 Decoder의 Self-Attention에는 **대각선 아래(자기 자신 포함 과거)만 보이고, 대각선 위(미래)는 가리는** 삼각형 모양의 마스크가 하나 더 필요하다 — Look-ahead Mask, 또는 Causal Mask라고 부른다.

가리는 방법은 Padding Mask와 완전히 동일하다: `masked_fill(mask == 0, -inf)` 후 `softmax`.

In [ ]:
# 대각선을 포함해서 아래쪽(과거+현재)만 True로 남기는 하삼각행렬
def make_causal_mask(seq_len):
    mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
    return mask   # [seq_len, seq_len]

print(make_causal_mask(4))
# tensor([[ True, False, False, False],
#         [ True,  True, False, False],
#         [ True,  True,  True, False],
#         [ True,  True,  True,  True]])

> 실전에서는 Padding Mask와 Causal Mask를 `&`(AND) 연산으로 합쳐서 한 번에 적용한다 — 둘 중 하나라도 가려야 하는 위치면 가린다.

### 이전에 구현한 PositionalEncoding, scaled_dot_product_attention 복습

이번에도 그대로 재사용한다 (3주차 ~ 5주차와 동일). `scaled_dot_product_attention`은 원래부터 `mask` 인자를 받고 있었으므로, 이제부터 Padding Mask든 Causal Mask든 그대로 넘기기만 하면 된다.

In [ ]:
import math

# 3주차 ~ 5주차에서 구현한 Positional Encoding 그대로 재사용
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        T = x.size(1)

        return x + self.pe[:, :T]

# mask는 Padding Mask([B,1,1,T])든 Causal Mask([T,T])든 그대로 브로드캐스팅됨
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    weights = torch.softmax(scores, dim=-1)
    output = weights @ V

    return output, weights

# MultiHead Attention 일반화 — Cross-Attention을 위한 준비

4~5주차 버전은 입력 `x` 하나에서 Q, K, V를 모두 만들었다. 하지만 06번에서 만들 **Cross-Attention**은 Query와 Key·Value의 출처가 다르다 (Query는 Decoder, Key·Value는 Encoder의 출력). 그래서 인자를 `query, key, value`로 분리해둔다.

In [ ]:
# MultiHead Attention Layer — query/key/value를 분리해서 Cross-Attention도 처리 가능하도록 일반화
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.h = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, T, _ = x.shape
        x = x.view(B, T, self.h, self.d_k)

        return x.transpose(1, 2)  # [B, h, T, d_k]

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))

        out, weights = scaled_dot_product_attention(Q, K, V, mask)

        out = out.transpose(1, 2).contiguous().view(B, -1, self.h * self.d_k)

        return self.W_o(out)

Self-Attention은 그냥 `query=key=value=x`로 호출하면 된다 — 완전히 같은 클래스를 그대로 재사용한다.

# Encoder Layer — 이번엔 진짜로 Mask를 적용

4~5주차의 EncoderLayer는 `mask` 인자를 받아만 두고 실제로 넘겨준 적은 없었다. 이번엔 `<pad>` 위치를 실제로 가리기 위해 Padding Mask를 만들어서 넘긴다.

In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)   # [B, T, d_model] → 동일 shape


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-Attention이므로 query=key=value=x, mask에는 Padding Mask를 넘김
        attn_out = self.mha(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x

In [ ]:
# <pad> 토큰(0)이 섞인 입력으로 mask가 실제로 shape을 깨지 않고 잘 동작하는지 확인
seq = torch.tensor([[5, 8, 3, 0, 0], [1, 2, 3, 4, 6]])
pad_mask = make_padding_mask(seq)

encoder_layer = EncoderLayer(d_model=128, num_heads=8, d_ff=512)
x = torch.randn(2, 5, 128)
print(encoder_layer(x, pad_mask).shape)   # torch.Size([2, 5, 128]) — shape은 그대로!

# Encoder Stack

4~5주차와 동일하게 EncoderLayer를 N개 쌓으면 Encoder 전체가 된다. 다만 **여러 층을 깊게 쌓은 최종 완성형은 06번**에서 전체 Transformer와 함께 만들 것이므로, 여기서는 구조만 확인하는 용도로 **1개 층만** 쌓는다.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)   # 이전 layer의 출력이 다음 layer의 입력

        return x

In [ ]:
x = torch.randn(2, 10, 512)   # [batch=2, seq_len=10, d_model=512]
encoder = Encoder(num_layers=1, d_model=512, num_heads=8, d_ff=2048)   # 06번 전이라 1개 층만

out = encoder(x)
print(out.shape)   # torch.Size([2, 10, 512]) — 입력과 동일!

# Decoder Layer

Encoder는 입력 문장 전체를 한꺼번에 봐도 됐지만, Decoder는 "이전 단어들로 다음 단어를 하나씩 생성"하는 역할이다. 그래서 Decoder Layer는 Encoder Layer보다 서브레이어가 하나 더 있다.

1. **Masked Multi-Head Self-Attention** — 지금까지 생성된 단어들끼리만, Look-ahead Mask를 적용해서 Attention
2. **Multi-Head Cross-Attention** — Query는 Decoder에서, **Key와 Value는 Encoder의 출력**에서 가져옴
3. **Feed Forward Network** — Encoder와 동일한 position-wise FFN

세 서브레이어 모두 Residual Connection + Layer Norm이 붙는다 (5주차에서 만든 패턴 그대로).

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, causal_mask=None, pad_mask=None):
        # 1) Masked Self-Attention: query=key=value=x, 미래를 가리는 causal_mask
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, causal_mask)))

        # 2) Cross-Attention: query=Decoder(x), key=value=Encoder 출력(enc_out)
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, pad_mask)))

        # 3) Feed Forward
        x = self.norm3(x + self.dropout(self.ffn(x)))

        return x

In [ ]:
decoder_layer = DecoderLayer(d_model=128, num_heads=8, d_ff=512)
x = torch.randn(2, 7, 128)         # decoder 입력
enc_out = torch.randn(2, 10, 128)  # encoder 출력
cmask = make_causal_mask(7)

print(decoder_layer(x, enc_out, cmask, None).shape)   # torch.Size([2, 7, 128])

# Decoder Stack

Encoder와 마찬가지로 DecoderLayer를 N개 쌓으면 Decoder 전체가 된다. 단, 매 layer마다 **같은 enc_out(Encoder의 최종 출력)**을 계속 Cross-Attention에 전달한다는 점이 다르다.

여기서도 최종 완성형은 06번에서 만들 것이므로, 우선 **1개 층만** 쌓아서 구조를 확인한다.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, enc_out, causal_mask=None, pad_mask=None):
        for layer in self.layers:
            x = layer(x, enc_out, causal_mask, pad_mask)   # 모든 layer가 같은 enc_out을 참고

        return x

In [ ]:
decoder = Decoder(num_layers=1, d_model=512, num_heads=8, d_ff=2048)   # 06번 전이라 1개 층만
x = torch.randn(2, 7, 512)         # decoder 입력
enc_out = torch.randn(2, 10, 512)  # encoder 출력
cmask = make_causal_mask(7)

out = decoder(x, enc_out, cmask, None)
print(out.shape)   # torch.Size([2, 7, 512]) — 입력과 동일!

# FULL Code

Raw Data -> Tokenizer -> Padding or Truncate -> Embedding + Positional Encoding -> **Encoder(Multi-Head Attention + Feed Forward, Padding Mask 적용, 1 Layer)**

감정 분류는 문장 하나를 보고 라벨 하나를 맞추는 작업이라 Decoder(Cross-Attention)까지는 필요 없다. 그래서 이번에도 4~5주차와 같은 감정 분류 모델을 그대로 확장하되, **Encoder의 Self-Attention에 실제로 Padding Mask를 적용**해서 `<pad>` 토큰이 결과에 영향을 주지 않도록 만든다.

(Decoder까지 합쳐 번역 같은 seq2seq 문제를 푸는 전체 Transformer는 06번에서 완성한다.)

In [ ]:
# 감정 분류 모델에 Padding Mask를 적용한 버전
class TSModel(nn.Module):
    def __init__(self, vocab_size=50000, embed_dim=128, num_classes=2, num_layers=1, num_heads=8, d_ff=512, pad_idx=0):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.pad_idx = pad_idx

        # Embedding
        self.embed_layer = nn.Embedding(vocab_size, embed_dim)

        # Positional Encoding
        self.pe_layer = PositionalEncoding(d_model=embed_dim)

        # 🍏🍏
        # Encoder Stack (06번에서 최종 완성될 예정이라 우선 1개 층만 쌓음)
        self.encoder = Encoder(num_layers=num_layers, d_model=embed_dim, num_heads=num_heads, d_ff=d_ff)

        # 감정 분류(클래스가 2개)이므로 마지막 feed forward만 넣어주면 우리가 원하는 모델을 만들 수 있음.
        self.feed_forward = nn.Linear(embed_dim, num_classes)

        # softmax
        self.act = nn.Softmax(dim=-1)

    def forward(self, x):
        # 🍏🍏
        # <pad> 위치를 가리는 Padding Mask (입력 토큰 id에서 바로 만듦)
        pad_mask = make_padding_mask(x, pad_idx=self.pad_idx)
        print(pad_mask.shape)

        # Embedding
        embeded_x = self.embed_layer(x)

        # Positional Encoding
        pe_embeded_x = self.pe_layer(embeded_x)
        print(pe_embeded_x.shape)

        # 🍏🍏
        # Encoder Stack — 이번엔 pad_mask를 실제로 넘겨서 <pad> 토큰을 무시하게 만듦
        output = self.encoder(pe_embeded_x, pad_mask)
        print(output.shape)

        pooled = output.mean(dim=1)
        print(pooled.shape)

        # 긍정인지 부정인지 확인
        x = self.feed_forward(pooled)
        print(x.shape)

        # 확률값으로 변환
        x = self.act(x)

        return x

In [ ]:
ts_model = TSModel()

In [ ]:
x = torch.tensor([[1, 2, 3, 4]])
probas = ts_model(x)
print(probas)

# 감정 분류 학습 시켜보기

Naver에서 제공한 영화 리뷰 감정 분류 데이터셋(NSMC)
* 0이 부정
* 1이 긍정

학습·추론 코드는 4~5주차와 **거의 동일**하다 — 달라지는 건 모델 내부에서 Padding Mask를 만들어 Encoder에 넘겨준다는 것뿐이다.

### 0. Utils

In [ ]:
# vocab 만드는 함수
# setences는 vocab을 만들 때 필요한 데이터
def create_vocab(sentences, special_tokens=["<pad>", "<sos>", "<eos>", "<unk>"]):
    words = sorted(set(w for s in sentences for w in s.split()))
    vocab = special_tokens + words  # 특수 토큰들에 대해서도 index를 매기기 위해서 SPECIALS와 words를 합침.
    stoi = {w: i for i, w in enumerate(vocab)}
    itos = {i: w for i, w in enumerate(vocab)}

    return words, vocab, stoi, itos

In [ ]:
import torch

# 앞에서 encode 함수는 string에 대해서 처리
def encode(sentence, stoi):
    # 예외처리가 동작하면서 이전에 없던 값들이면 <unk>로 대체.
    ids = [stoi.get(w, stoi["<unk>"]) for w in sentence.split()]
    return [stoi["<sos>"]] + ids + [stoi["<eos>"]]

# 실제로는 리스트 형태로 들어옴(텐서)
# 이에 맞춰서 preprocessing 함수를 새로 선언
def preprocessing(text, stoi, padding_size=10, padding_token=0):
    input_tokens = []
    for sentence in text:
        # 우선 각 토큰들을 encoding
        encoded_sentence = encode(sentence, stoi)

        # embedding에 들어갈 때는 모든 데이터의 크기가 동일해야 하므로, padding적용
        encoded_padded_sentence = encoded_sentence + [padding_token] * (padding_size - len(encoded_sentence))

        input_tokens.append(encoded_padded_sentence)

    return torch.tensor(input_tokens)  # tensor 적용은 반드시 해야함.

# vocab에서 "<pad>"는 항상 index 0이므로, padding_token=0은 곧 make_padding_mask의 pad_idx=0과 그대로 맞아떨어진다.

### 1. Load Data

In [ ]:
import pandas as pd

train_url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
test_url  = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"

train_df = pd.read_csv(train_url, sep="\t")
test_df  = pd.read_csv(test_url, sep="\t")

print(train_df.head())

In [ ]:
# NAN값 제거
train_df = train_df.dropna(subset=["document"])
test_df  = test_df.dropna(subset=["document"])

train_texts, train_labels = train_df["document"].tolist(), train_df["label"].tolist()
test_texts, test_labels   = test_df["document"].tolist(), test_df["label"].tolist()

### 2. Vocab 생성

In [ ]:
# vocab 사전을 만들기 위한 데이터
sentences = train_texts

# 특수 토큰: 패딩, 문장 시작/끝, 미등록 단어
special_tokens = ["<pad>", "<sos>", "<eos>", "<unk>"]

words, vocab, stoi, itos = create_vocab(sentences[:10000], special_tokens)

### 3. Model 생성

In [ ]:
# 학습에 사용할 감정 분류 모델 (디버그용 print는 제거)
class TSModel(nn.Module):
    def __init__(self, vocab_size=50000, embed_dim=128, num_classes=2, num_layers=1, num_heads=8, d_ff=512, pad_idx=0):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.pad_idx = pad_idx

        # Embedding
        self.embed_layer = nn.Embedding(vocab_size, embed_dim)

        # Positional Encoding
        self.pe_layer = PositionalEncoding(d_model=embed_dim)

        # 🍏🍏
        # Encoder Stack (06번에서 최종 완성될 예정이라 우선 1개 층만 쌓음)
        self.encoder = Encoder(num_layers=num_layers, d_model=embed_dim, num_heads=num_heads, d_ff=d_ff)

        # 감정 분류(클래스가 2개)이므로 마지막 feed forward만 넣어주면 우리가 원하는 모델을 만들 수 있음.
        self.feed_forward = nn.Linear(embed_dim, num_classes)

        # softmax
        self.act = nn.Softmax(dim=-1)

    def forward(self, x):
        # 🍏🍏
        # <pad> 위치를 가리는 Padding Mask
        pad_mask = make_padding_mask(x, pad_idx=self.pad_idx)

        # Embedding
        embeded_x = self.embed_layer(x)

        # Positional Encoding
        pe_embeded_x = self.pe_layer(embeded_x)

        # 🍏🍏
        # Encoder Stack — <pad> 토큰은 pad_mask로 가려서 Attention에서 무시됨
        output = self.encoder(pe_embeded_x, pad_mask)

        pooled = output.mean(dim=1)

        # 긍정인지 부정인지 확인
        x = self.feed_forward(pooled)

        # 확률값으로 변환
        x = self.act(x)

        return x

In [ ]:
# 감정 분류 모델 생성
set_model = TSModel(vocab_size=len(vocab))

### 4. DataSet과 DataLoader 생성

In [ ]:
from torch.utils.data import Dataset, DataLoader

# 여기서 transform은 preprocessing을 의미
class NSMC_Dataset(Dataset):
    def __init__(self, texts, labels, stoi=None, transform=None):
        self.texts = texts
        self.labels = labels
        self.stoi = stoi
        self.transform = transform

    def __len__(self):
        # 원래 tensor로 들어오면 shape이나 size 함수로 선언 가능하지만, 리스트가 들어오므로 len으로 호출
        return len(self.texts)

    def __getitem__(self, idx):
        batch_texts = self.texts[idx]
        batch_labels = self.labels[idx]

        if self.transform:
            # preprocessing은 문장 리스트를 받아 [N, padding_size] 텐서를 반환하므로
            # 문장 1개도 리스트로 감싸서 넣고, 결과에서 배치 차원([0])을 벗겨 [padding_size]로 되돌림.
            batch_texts = self.transform([batch_texts], stoi, padding_size=50)[0]

        return batch_texts, batch_labels

In [ ]:
train_dataset = NSMC_Dataset(train_texts, train_labels, stoi, preprocessing)
test_dataset = NSMC_Dataset(test_texts, test_labels, stoi, preprocessing)

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
next(iter(train_dataloader))[0].shape

In [ ]:
output = set_model(next(iter(train_dataloader))[0])

### 5. 학습

In [ ]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

# GPU 사용 가능하면 GPU, 아니면 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 위에서 이미 만들어둔 것들을 그대로 사용
# - set_model: TSModel(vocab_size=50000, embed_dim=128, num_classes=2, num_layers=1, ...) — Padding Mask 적용됨
# - train_dataloader, test_dataloader: NSMC 데이터를 stoi로 인코딩한 DataLoader
set_model = set_model.to(device)

# set_model의 forward가 마지막에 Softmax(dim=-1)로 "확률"을 반환하므로,
# 내부에서 log_softmax를 또 적용하는 CrossEntropyLoss 대신 NLLLoss(log(확률))를 사용.
criterion = nn.NLLLoss()
optimizer = torch.optim.Adam(set_model.parameters(), lr=1e-3)

# tqdm 갱신 주기 (20 batch마다 한 번씩만 postfix 갱신)
LOG_INTERVAL = 20

# Train 및 Eval 로직
num_epochs = 5
for epoch in range(num_epochs):
    # 1. model train
    set_model.train()
    train_loss = 0.0
    train_preds_all, train_labels_all = [], []

    with tqdm(train_dataloader, desc=f"[Epoch {epoch+1}/{num_epochs}] Train", leave=False) as train_bar:
        for batch_idx, (texts, labels) in enumerate(train_bar):
            texts, labels = texts.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            # 1-1. model 예측
            # set_model(texts)는 forward 내부에서 Padding Mask를 만들어 Encoder에 넘기고,
            # 시퀀스 축을 pooling해서 [B, num_classes] 확률을 반환함.
            probs = set_model(texts)  # [B, num_classes]

            # 1-2. loss function 계산
            loss = criterion(torch.log(probs.clamp_min(1e-9)), labels)

            # 1-3. 이전 기울기 초기화
            optimizer.zero_grad()

            # 1-4. 역전파(backpropagation) -> 기울기 계산
            loss.backward()

            # 1-5. 파라미터 업데이트
            optimizer.step()

            train_loss += loss.item() * texts.size(0)
            train_preds_all.extend(probs.argmax(dim=1).cpu().tolist())
            train_labels_all.extend(labels.cpu().tolist())

            # 20 batch마다 한 번씩 loss/accuracy 갱신
            if (batch_idx + 1) % LOG_INTERVAL == 0:
                train_accuracy = accuracy_score(train_labels_all, train_preds_all)
                train_bar.set_postfix(loss=f"{loss.item():.4f}", accuracy=f"{train_accuracy:.2%}")

    train_loss /= len(train_dataloader.dataset)
    train_accuracy = accuracy_score(train_labels_all, train_preds_all)

    # 2. model eval
    set_model.eval()
    eval_preds_all, eval_labels_all = [], []
    accuracy = 0.0

    with torch.no_grad(), tqdm(test_dataloader, desc=f"[Epoch {epoch+1}/{num_epochs}] Eval", leave=False) as eval_bar:
        for batch_idx, (texts, labels) in enumerate(eval_bar):
            texts, labels = texts.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            probs = set_model(texts)  # [B, num_classes]
            eval_preds_all.extend(probs.argmax(dim=1).cpu().tolist())
            eval_labels_all.extend(labels.cpu().tolist())

            # 20 batch마다 한 번씩 accuracy 갱신
            if (batch_idx + 1) % LOG_INTERVAL == 0:
                accuracy = accuracy_score(eval_labels_all, eval_preds_all)
                eval_bar.set_postfix(accuracy=f"{accuracy:.2%}")

    accuracy = accuracy_score(eval_labels_all, eval_preds_all)

    print(f"epoch {epoch+1}/{num_epochs} | train_loss {train_loss:.4f} | train_accuracy {train_accuracy:.2%} | accuracy {accuracy:.2%}")

# 모델 저장
model_path = "ts_model_mask_attention_nsmc.pth"
torch.save(set_model.state_dict(), model_path)
print(f"모델 저장 완료: {model_path}")

### 6. 예측

실제 모델은 당연히 숫자값들에 대해 추론을 하지만, 여기서는 한국말이 나오면 그대로 나오도록 하는 함수를 작성

In [ ]:
def predict(set_model, text, stoi, padding_size=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    preprocessed_text = preprocessing([text], stoi=stoi, padding_size=padding_size)
    preprocessed_text = preprocessed_text.to(device)

    probs = set_model(preprocessed_text)  # [1, num_classes]

    pred = probs.argmax(dim=-1).item()

    itos = ["부정", "긍정"]

    return itos[pred]

In [ ]:
pred = predict(set_model, "이거 좋다.", stoi, padding_size=50)

In [ ]:
pred

In [ ]:
while True:
    text = input("🫠 텍스트를 입력하세요: ")
    if text == "exit":
        break
    print(f"User: {text}")

    pred = predict(set_model, text, stoi, padding_size=50)

    print(f">> 당신이 하시는 말씀은 {pred}적인 어투입니다 !")

### 오늘 배운 것 & 다음 예고

* Padding Mask = `<pad>` 무시, Causal Mask = 미래 차단
* `masked_fill(mask==0, -inf)` 후 `softmax`
* MultiHeadAttention을 query/key/value 분리 형태로 일반화 → Self-Attention도, Cross-Attention도 같은 클래스로 처리
* DecoderLayer = Masked Self-Attention + Cross-Attention + FFN (Add & Norm 3번)
* Encoder / Decoder 모두 1개 층만 쌓아서 구조만 확인 (완전한 버전은 06번에서)

**06번 예고**: Encoder + Decoder를 하나로 합쳐 전체 Transformer를 완성하고, Output Linear + Softmax로 다음 단어 확률까지 만들어본다.